# 01 - Prepare the RoadWatcher data

Run this notebook from top to bottom. It copies the source data, checks labels and images, creates the fixed test split, and writes `roadwatcher.yaml`.

In [2]:
from pathlib import Path
import hashlib
import random
import shutil
from collections import Counter
import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageDraw

NOTEBOOK_DIR = Path.cwd()
PROJECT = NOTEBOOK_DIR.parent
SOURCE_DIR = PROJECT / 'trafic_data'
DATA_DIR = PROJECT / 'data_clean'
REMOVE_DUPLICATES = False
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
EXPECTED = {'train': 2297, 'valid': 298, 'test': 405}

if not DATA_DIR.exists():
    shutil.copytree(SOURCE_DIR, DATA_DIR, ignore=shutil.ignore_patterns('*.cache'))

def get_images(split):
    return sorted(p for p in (DATA_DIR / split / 'images').iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)

def get_labels(split):
    return sorted((DATA_DIR / split / 'labels').glob('*.txt'))

def read_label(path):
    return [line.split() for line in path.read_text().splitlines() if line.strip()]

def is_valid_label(row, class_count):
    if len(row) != 5:
        return False
    try:
        cls, x, y, width, height = map(float, row)
    except ValueError:
        return False
    return cls.is_integer() and 0 <= int(cls) < class_count and all(0 <= value <= 1 for value in (x, y, width, height)) and width > 0 and height > 0

def delete_sample(image, label):
    image.unlink(missing_ok=True)
    label.unlink(missing_ok=True)

def image_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

FileNotFoundError: [Errno 2] No such file or directory: '/trafic_data'

## Check and clean
Empty or invalid labels and unreadable images are removed together with their image/label pair.

In [ ]:
source_config = yaml.safe_load((SOURCE_DIR / 'data_1.yaml').read_text())
CLASS_NAMES = source_config['names']
CLASS_COUNT = int(source_config['nc'])

for split in ('train', 'valid', 'test'):
    image_dir = DATA_DIR / split / 'images'
    label_dir = DATA_DIR / split / 'labels'
    image_names = {p.stem for p in get_images(split)}
    label_names = {p.stem for p in get_labels(split)}
    for missing in sorted(image_names - label_names):
        (image_dir / next(p.name for p in get_images(split) if p.stem == missing)).unlink(missing_ok=True)
    for missing in sorted(label_names - image_names):
        (label_dir / f'{missing}.txt').unlink(missing_ok=True)
    for image in get_images(split):
        label = label_dir / f'{image.stem}.txt'
        rows = read_label(label)
        if not rows or any(not is_valid_label(row, CLASS_COUNT) for row in rows):
            delete_sample(image, label)
            continue
        try:
            with Image.open(image) as checked:
                checked.verify()
        except Exception:
            delete_sample(image, label)

for split in ('train', 'valid', 'test'):
    assert {p.stem for p in get_images(split)} == {p.stem for p in get_labels(split)}, split
print('Image/label names match after cleaning.')

## Exact duplicates
The default is report-only because the project requires 2,297 training images and the source contains one duplicate pair inside train.

In [ ]:
def find_duplicates():
    groups = {}
    for split in ('train', 'valid', 'test'):
        for image in get_images(split):
            groups.setdefault(image_hash(image), []).append((split, image))
    return [items for items in groups.values() if len(items) > 1]

duplicates = find_duplicates()
print('Exact duplicate groups:', [[f'{split}/{path.name}' for split, path in group] for group in duplicates])
if REMOVE_DUPLICATES:
    for group in duplicates:
        for _, image in group[1:]:
            delete_sample(image, DATA_DIR / _ / 'labels' / f'{image.stem}.txt')

## Fixed test split
A sorted training file list and `random.Random(42)` make this reproducible. If test already contains images, the split is left untouched.

In [ ]:
test_images = get_images('test')
if not test_images:
    candidates = get_images('train')
    random.Random(42).shuffle(candidates)
    for image in candidates[:405]:
        label = DATA_DIR / 'train' / 'labels' / f'{image.stem}.txt'
        shutil.move(str(image), DATA_DIR / 'test' / 'images' / image.name)
        shutil.move(str(label), DATA_DIR / 'test' / 'labels' / label.name)

counts = {split: len(get_images(split)) for split in ('train', 'valid', 'test')}
print(counts)
assert counts == EXPECTED, f'Expected {EXPECTED}, got {counts}'

hashes = {}
for split in counts:
    for image in get_images(split):
        hashes.setdefault(image_hash(image), []).append(split)
cross_split = [splits for splits in hashes.values() if len(set(splits)) > 1]
print('Cross-split exact duplicate groups:', len(cross_split))
assert not cross_split

## Dataset YAML and exploration

In [ ]:
yaml_path = PROJECT / 'roadwatcher.yaml'
yaml_path.write_text(yaml.safe_dump({'path': str(DATA_DIR.resolve()), 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': CLASS_COUNT, 'names': CLASS_NAMES}, sort_keys=False))
print(yaml_path)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
samples = get_images('train')[:6]
for axis, image in zip(axes.flat, samples):
    with Image.open(image).convert('RGB') as opened:
        picture = opened.copy()
    draw = ImageDraw.Draw(picture)
    width, height = picture.size
    for row in read_label(DATA_DIR / 'train' / 'labels' / f'{image.stem}.txt'):
        cls, x, y, box_width, box_height = map(float, row)
        x1, y1 = (x - box_width / 2) * width, (y - box_height / 2) * height
        x2, y2 = (x + box_width / 2) * width, (y + box_height / 2) * height
        draw.rectangle((x1, y1, x2, y2), outline='red', width=3)
        draw.text((x1, y1), CLASS_NAMES[int(cls)], fill='yellow')
    axis.imshow(picture)
    axis.axis('off')
plt.tight_layout()

class_counts = Counter()
for split in ('train', 'valid', 'test'):
    for label in get_labels(split):
        class_counts.update(int(row[0]) for row in read_label(label))
table = {CLASS_NAMES[index]: class_counts[index] for index in range(CLASS_COUNT)}
print(table)
plt.figure(figsize=(12, 4))
plt.bar(table.keys(), table.values())
plt.xticks(rotation=75, ha='right')
plt.ylabel('Objects')
plt.tight_layout()